<a href="https://colab.research.google.com/github/Huang-stat/Computer-Labs/blob/main/AKHU_CS_Lab3_0508.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS Lab 3: Custom Exceptions and Fault-Tolerant Design

**Topic:** EAFP style, custom exceptions, reliable data loading, and simple summaries before modeling.  
**Main rule for this lab:** no external packages are required.

This lab slows down because reliable data loading comes before baseline models, regression models, model comparison, and evaluation metrics. Today, the goal is to load and summarize data safely using built-in Python.

## 0. Roadmap

`open file -> read rows -> check columns -> convert numbers -> summarize data -> report clear errors`

### Your turn
Write one possible mistake for each step. For example, the file may be missing, a column name may be wrong, or a value such as `NA` may not convert to a number.

In [1]:
# Built-in modules only: no pip install needed.
from pathlib import Path
import csv
from statistics import mean, median

## 1. Create or locate the simple CSV file

The notebook first looks for a file named `AKHU_student_scores_lab3_simple.csv` in the `data/` folder. If it is missing, the notebook creates a small CSV file automatically.

In [2]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
DATA_PATH = DATA_DIR / "AKHU_student_scores_lab3_simple.csv"

SAMPLE_ROWS_FULL = [
    {"student_id": "S01", "hours_studied": "1.5", "attendance_rate": "0.55", "assignments_submitted": "4", "quiz_average": "48", "final_score": "52"},
    {"student_id": "S02", "hours_studied": "2.0", "attendance_rate": "0.60", "assignments_submitted": "5", "quiz_average": "50", "final_score": "56"},
    {"student_id": "S03", "hours_studied": "2.3", "attendance_rate": "0.66", "assignments_submitted": "5", "quiz_average": "54", "final_score": "59"},
    {"student_id": "S04", "hours_studied": "2.7", "attendance_rate": "0.70", "assignments_submitted": "6", "quiz_average": "58", "final_score": "63"},
    {"student_id": "S05", "hours_studied": "3.0", "attendance_rate": "0.74", "assignments_submitted": "7", "quiz_average": "62", "final_score": "67"},
    {"student_id": "S06", "hours_studied": "3.3", "attendance_rate": "0.78", "assignments_submitted": "7", "quiz_average": "66", "final_score": "71"},
    {"student_id": "S07", "hours_studied": "3.5", "attendance_rate": "0.80", "assignments_submitted": "8", "quiz_average": "68", "final_score": "73"},
    {"student_id": "S08", "hours_studied": "3.8", "attendance_rate": "0.84", "assignments_submitted": "8", "quiz_average": "72", "final_score": "76"},
    {"student_id": "S09", "hours_studied": "4.0", "attendance_rate": "0.86", "assignments_submitted": "9", "quiz_average": "74", "final_score": "79"},
    {"student_id": "S10", "hours_studied": "4.2", "attendance_rate": "0.88", "assignments_submitted": "9", "quiz_average": "78", "final_score": "83"},
    {"student_id": "S11", "hours_studied": "4.4", "attendance_rate": "0.90", "assignments_submitted": "9", "quiz_average": "81", "final_score": "87"},
    {"student_id": "S12", "hours_studied": "2.2", "attendance_rate": "0.64", "assignments_submitted": "5", "quiz_average": "52", "final_score": "57"},
]

if not DATA_PATH.exists():
    with open(DATA_PATH, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(SAMPLE_ROWS_FULL[0].keys()))
        writer.writeheader()
        writer.writerows(SAMPLE_ROWS_FULL)

print("CSV path:", DATA_PATH)
print("File exists:", DATA_PATH.exists())

CSV path: data/AKHU_student_scores_lab3_simple.csv
File exists: True


## 2. EAFP style

EAFP means **Easier to Ask Forgiveness than Permission**. Try the risky operation first. If it fails, catch the specific exception.

### Your turn
Change `text = "73.5"` to `text = "NA"`. What changes?

In [3]:
text = "73.5"

try:
    number = float(text)
    print("Converted number:", number)
except ValueError:
    print("Could not convert this value to a number:", text)

Converted number: 73.5


## 3. Custom exceptions

A custom exception gives a clear name to a problem from this lab. These names are more helpful than using one generic `Exception` for every problem.

In [4]:
class LabDataError(Exception):
    pass

class MissingFileError(LabDataError):
    pass

class EmptyDatasetError(LabDataError):
    pass

class MissingColumnError(LabDataError):
    pass

class NumericConversionError(LabDataError):
    pass

### Hint with code: raise a custom exception

This example intentionally asks for a column that is not in the row.

In [5]:
required_column = "quiz_average"
row = {"final_score": "82"}

try:
    value = row[required_column]
except KeyError as exc:
    print("Controlled error message:")
    print(f"Required column '{required_column}' was not found.")

Controlled error message:
Required column 'quiz_average' was not found.


## 4. Load CSV with EAFP

The risky step is opening the file. A missing file should become a clear `MissingFileError`.

In [6]:
def load_csv_eafp(path):
    try:
        with open(path, newline="", encoding="utf-8") as file:
            reader = csv.DictReader(file)
            rows = list(reader)
    except FileNotFoundError as exc:
        raise MissingFileError(f"Could not find file: {path}") from exc

    if not rows:
        raise EmptyDatasetError(f"File has no data rows: {path}")

    return rows

rows = load_csv_eafp(DATA_PATH)
print("Number of rows:", len(rows))
print("First row:", rows[0])

Number of rows: 12
First row: {'student_id': 'S01', 'hours_studied': '1.5', 'attendance_rate': '0.55', 'assignments_submitted': '4', 'quiz_average': '48', 'final_score': '52'}


## 5. Safe classroom fallback

For class practice, a fallback can keep the notebook running. The fallback must still explain what happened. Do not silently hide data problems.

In [7]:
SAMPLE_ROWS_SMALL = [
    {"student_id": "S01", "hours_studied": "1.5", "quiz_average": "48", "final_score": "52"},
    {"student_id": "S02", "hours_studied": "2.0", "quiz_average": "50", "final_score": "56"},
]

def load_or_use_sample(path):
    try:
        rows = load_csv_eafp(path)
        print(f"Loaded {len(rows)} rows from {path}")
        return rows
    except LabDataError as exc:
        print(f"Using built-in sample data because: {exc}")
        return SAMPLE_ROWS_SMALL.copy()

rows_from_fallback = load_or_use_sample(Path("missing_file.csv"))
rows_from_fallback

Using built-in sample data because: Could not find file: missing_file.csv


[{'student_id': 'S01',
  'hours_studied': '1.5',
  'quiz_average': '48',
  'final_score': '52'},
 {'student_id': 'S02',
  'hours_studied': '2.0',
  'quiz_average': '50',
  'final_score': '56'}]

## 6. Validate required columns and missing values

Before summarizing or modeling, check that required columns exist and that important cells are not missing.

In [8]:
MISSING_MARKERS = {"", "NA", "N/A", "None", "nan", "NaN"}

REQUIRED_COLUMNS = [
    "student_id",
    "hours_studied",
    "quiz_average",
    "final_score",
]

def is_missing(value):
    return value is None or str(value).strip() in MISSING_MARKERS


def validate_columns(rows, required_columns):
    if not rows:
        raise EmptyDatasetError("No rows are available for validation.")

    first_row = rows[0]
    for column in required_columns:
        if column not in first_row:
            raise MissingColumnError(f"Missing required column: {column}")


def validate_no_missing(rows, required_columns):
    for row_number, row in enumerate(rows, start=1):
        for column in required_columns:
            if is_missing(row.get(column)):
                raise LabDataError(
                    f"Missing value at row {row_number}, column '{column}'."
                )

rows = load_or_use_sample(DATA_PATH)
validate_columns(rows, REQUIRED_COLUMNS)
validate_no_missing(rows, REQUIRED_COLUMNS)
print("Validation passed.")

Loaded 12 rows from data/AKHU_student_scores_lab3_simple.csv
Validation passed.


### Hint with code: check missing values with pandas only if available

This is optional. The main lab does not require pandas.

In [9]:
try:
    import pandas as pd
    df = pd.read_csv(DATA_PATH)
    print(df.info())
    print(df.isna().sum())
except ImportError:
    print("pandas is not installed. That is OK for this lab.")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   student_id             12 non-null     object 
 1   hours_studied          12 non-null     float64
 2   attendance_rate        12 non-null     float64
 3   assignments_submitted  12 non-null     int64  
 4   quiz_average           12 non-null     int64  
 5   final_score            12 non-null     int64  
dtypes: float64(2), int64(3), object(1)
memory usage: 708.0+ bytes
None
student_id               0
hours_studied            0
attendance_rate          0
assignments_submitted    0
quiz_average             0
final_score              0
dtype: int64


## 7. Convert numeric columns safely

CSV values are text. The string `"82"` must become the number `82.0` before computing a mean.

In [10]:
def get_float(row, column, row_number):
    try:
        return float(row[column])
    except KeyError as exc:
        raise MissingColumnError(
            f"Missing column '{column}' at row {row_number}."
        ) from exc
    except (TypeError, ValueError) as exc:
        bad_value = row.get(column)
        raise NumericConversionError(
            f"Row {row_number}, column '{column}' has non-numeric value: {bad_value!r}"
        ) from exc

print(get_float(rows[0], "final_score", 1))

52.0


## 8. Summarize the data

This lab stops at loading, checking, converting, and summarizing. Baseline models, regression models, model comparison, and evaluation metrics will come later.

In [11]:
def summarize_numeric_column(rows, column):
    values = [get_float(row, column, i) for i, row in enumerate(rows, start=1)]
    if not values:
        raise EmptyDatasetError(f"No numeric values found for column: {column}")

    return {
        "column": column,
        "count": len(values),
        "min": min(values),
        "max": max(values),
        "mean": mean(values),
        "median": median(values),
    }


def print_summary(summary):
    print(f"Column: {summary['column']}")
    print(f"Count:  {summary['count']}")
    print(f"Min:    {summary['min']:.2f}")
    print(f"Max:    {summary['max']:.2f}")
    print(f"Mean:   {summary['mean']:.2f}")
    print(f"Median: {summary['median']:.2f}")

for column in ["hours_studied", "quiz_average", "final_score"]:
    summary = summarize_numeric_column(rows, column)
    print_summary(summary)
    print("-" * 30)

Column: hours_studied
Count:  12
Min:    1.50
Max:    4.40
Mean:   3.08
Median: 3.15
------------------------------
Column: quiz_average
Count:  12
Min:    48.00
Max:    81.00
Mean:   63.58
Median: 64.00
------------------------------
Column: final_score
Count:  12
Min:    52.00
Max:    87.00
Mean:   68.58
Median: 69.00
------------------------------


## 9. Break the data on purpose

A fault-tolerant program should fail clearly. Try these examples and read the controlled error messages.

In [12]:
bad_rows_missing_column = [
    {"student_id": "S01", "hours_studied": "1.5", "final_score": "52"}
]

try:
    validate_columns(bad_rows_missing_column, REQUIRED_COLUMNS)
except LabDataError as exc:
    print("Controlled error:", exc)

Controlled error: Missing required column: quiz_average


In [13]:
bad_rows_bad_number = [
    {"student_id": "S01", "hours_studied": "many", "quiz_average": "48", "final_score": "52"}
]

try:
    summarize_numeric_column(bad_rows_bad_number, "hours_studied")
except LabDataError as exc:
    print("Controlled error:", exc)

Controlled error: Row 1, column 'hours_studied' has non-numeric value: 'many'


## 10. Practice tasks

1. Load the CSV with `load_csv_eafp(DATA_PATH)`.
2. Print the number of rows and the first row.
3. Add a new missing marker, such as `"missing"`, to `MISSING_MARKERS`.
4. Test `is_missing("missing")`.
5. Create a row where `final_score` is `"high"`. Confirm that `NumericConversionError` is raised.
6. Summarize `attendance_rate`.
7. Write one sentence: "Before modeling, my program should check ..."

### Hint code for task 6

In [14]:
summary = summarize_numeric_column(rows, "attendance_rate")
print_summary(summary)

Column: attendance_rate
Count:  12
Min:    0.55
Max:    0.90
Mean:   0.75
Median: 0.76
